In [3]:
import random
import os
import time
from IPython.display import clear_output

In [4]:
def printG(grid):
    for i in range(0, 9, 3):
        print(grid[i:i+3])
    print()

In [5]:
def clear_screen():
    # os.system('cls' if os.name == 'nt' else 'clear')
    os.system('clear')

In [6]:
class Puzzle():
    def __init__(self, grid, goal):
        """Initializes the puzzle with a starting grid and a goal state."""
        self.grid = grid[:]
        self.goal = goal

    def find_index(self, num):
        """Finds the index of a given number in the grid."""
        for i, val in enumerate(self.grid):
            if val == num:
                return i
        return -1

    def get_valid_moves(self):
        """Returns a list of valid moves ('up', 'down', 'left', 'right') for the empty tile."""
        i = self.find_index(0)
        valid_moves = []
        
        # Check for UP move
        if i >= 3:
            valid_moves.append("up")
            
        # Check for DOWN move
        if i <= 5:
            valid_moves.append("down")
            
        # Check for LEFT move
        if i % 3 != 0:
            valid_moves.append("left")
            
        # Check for RIGHT move
        if i % 3 != 2:
            valid_moves.append("right")
            
        return valid_moves

    def move(self, direction):
        """Moves the empty tile in the specified direction."""
        i = self.find_index(0)
        
        if direction not in self.get_valid_moves():
            return

        if direction == "up":
            swap_index = i - 3
        elif direction == "down":
            swap_index = i + 3
        elif direction == "left":
            swap_index = i - 1
        elif direction == "right":
            swap_index = i + 1
        
        # Swap the empty tile with the target tile
        self.grid[i], self.grid[swap_index] = self.grid[swap_index], self.grid[i]

In [13]:
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def ReflexAgent():
    """A simple reflex agent to solve the 8-puzzle."""
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    
    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        valid_moves = puzzle.get_valid_moves()
        
        chosen_move = random.choice(valid_moves)
        
        puzzle.move(chosen_move)
        moves_count += 1

        # clear_output(wait=True)
        # print(f"Move {moves_count}: Moved {chosen_move}")
        # printG(puzzle.grid)
        # time.sleep(0.5)
        
    print(f"Solved in {moves_count} moves!")

ReflexAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Solved in 305578 moves!


In [11]:
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]
def ModelAgent():
    """A model-based reflex agent that avoids immediately reversing its last move."""
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    
    last_move_undo = None
    
    opposites = {'up': 'down', 'down': 'up', 'left': 'right', 'right': 'left'}

    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        possible_moves = puzzle.get_valid_moves()
        
        if last_move_undo in possible_moves and len(possible_moves) > 1:
            possible_moves.remove(last_move_undo)

        chosen_move = random.choice(possible_moves)
        
        last_move_undo = opposites[chosen_move]
        
        puzzle.move(chosen_move)
        moves_count += 1

        # clear_output(wait=True)
        # print(f"Move {moves_count}: Agent chose to move {chosen_move}")
        # printG(puzzle.grid)
        # time.sleep(0.5)
        
    print(f"Solved in {moves_count} moves!")

# Run the model-based agent
ModelAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Solved in 31178 moves!


In [21]:
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def calculate_manhattan_distance(grid, goal):
    distance = 0
    for i in range(9):
        if grid[i] != 0:
            row_current, col_current = divmod(i, 3)
            row_goal, col_goal = divmod(goal.index(grid[i]), 3)
            distance += abs(row_current - row_goal) + abs(col_current - col_goal)
    return distance

def GoalBasedAgent():
    """
    A goal-based agent that uses Manhattan distance but adds a penalty
    for moves that would disrupt an already solved row or column.
    """
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    
    # The agent's memory of states it has already been in.
    visited = {tuple(puzzle.grid)}
    
    # A large penalty to discourage breaking solved parts.
    PENALTY = 100 

    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        possible_moves = puzzle.get_valid_moves()
        scored_moves = []

        # Check if major sections are solved before evaluating moves.
        # Note: We check against the goal state.
        top_row_solved = (puzzle.grid[0:3] == puzzle.goal[0:3])
        left_col_solved = (puzzle.grid[0] == puzzle.goal[0] and 
                           puzzle.grid[3] == puzzle.goal[3] and 
                           puzzle.grid[6] == puzzle.goal[6])

        for move in possible_moves:
            temp_grid = puzzle.grid[:]
            i = temp_grid.index(0) # Current position of the blank tile

            # Simulate the move to get the next state
            if move == "up": swap_index = i - 3
            elif move == "down": swap_index = i + 3
            elif move == "left": swap_index = i - 1
            else: swap_index = i + 1 # right
            temp_grid[i], temp_grid[swap_index] = temp_grid[swap_index], temp_grid[i]

            # Rule 1: Avoid states we have already visited.
            if tuple(temp_grid) in visited:
                continue

            # Rule 2: Calculate the primary heuristic (Manhattan distance)
            distance = calculate_manhattan_distance(temp_grid, puzzle.goal)
            penalty = 0
            
            # Rule 3: Apply penalties if the move breaks a solved structure.
            # Penalty for moving a piece OUT of a solved top row.
            if top_row_solved and i < 3 and move == 'down':
                penalty = PENALTY
            # Penalty for moving a piece INTO a solved top row from below.
            if top_row_solved and i >= 3 and move == 'up':
                penalty = PENALTY
            
            # Penalty for moving a piece OUT of a solved left column.
            if left_col_solved and i in [0, 3, 6] and move == 'right':
                penalty = PENALTY
            # Penalty for moving a piece INTO a solved left column from the right.
            if left_col_solved and i in [1, 4, 7] and move == 'left':
                penalty = PENALTY
            
            scored_moves.append((distance + penalty, move))

        # If there are no new states to explore, the agent is stuck.
        if not scored_moves:
            print("Agent is stuck! No new moves available.")
            return

        # Sort moves by their score (lower is better) and pick the best one
        scored_moves.sort()
        best_score, best_move = scored_moves[0]
        
        # Apply the best move found
        puzzle.move(best_move)
        visited.add(tuple(puzzle.grid))
        moves_count += 1

        clear_output(wait=True)
        move_info = f"Score: {best_score}"
        if best_score >= PENALTY:
            move_info += " (Penalty Applied!)" # Let us know when it's making a tough choice
            
        print(f"Move {moves_count}: Agent chose {best_move} ({move_info})")
        printG(puzzle.grid)
        time.sleep(0.5)
        
    print(f"Solved in {moves_count} moves!")

GoalBasedAgent()

Move 86: Agent chose up (Score: 4)
[2, 1, 0]
[3, 4, 5]
[7, 6, 8]

Agent is stuck! No new moves available.


In [17]:
def calculate_utility(grid, goal, moves_count):
    """Calculates a utility score based on Manhattan distance and move count."""
    manhattan_distance = calculate_manhattan_distance(grid, goal)
    # Adjust the factor (0.001) to balance distance vs. move count
    utility = 1.0 / (manhattan_distance + 1) - (moves_count * 0.001) 

    return utility

def UtilityBasedAgent():
    """A utility-based agent that maximizes a utility score."""
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    
    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        possible_moves = puzzle.get_valid_moves()
        
        best_move = None
        max_utility = float('-inf')

        for move in possible_moves:
            temp_grid = puzzle.grid[:]

            i = temp_grid.index(0)
            if move == "up":
                swap_index = i - 3
            elif move == "down":
                swap_index = i + 3
            elif move == "left":
                swap_index = i - 1
            elif move == "right":
                swap_index = i + 1
            temp_grid[i], temp_grid[swap_index] = temp_grid[swap_index], temp_grid[i]

            utility = calculate_utility(temp_grid, puzzle.goal, moves_count + 1)
            

            if utility > max_utility:
                max_utility = utility
                best_move = move

        puzzle.move(best_move)
        moves_count += 1

        clear_output(wait=True)
        print(f"Move {moves_count}: Agent chose to move {best_move} (Utility: {max_utility})")
        printG(puzzle.grid)
        time.sleep(0.5)
        
    print(f"Solved in {moves_count} moves!")

UtilityBasedAgent()

Move 25: Agent chose to move down (Utility: 0.03055555555555555)
[7, 2, 4]
[0, 3, 6]
[5, 8, 1]



KeyboardInterrupt: 